# Insurance charges — EDA

Goal: predict `charges` (a customer's yearly insurance cost) from demographic and health features.
Quick exploratory pass before picking a model.

In [1]:
import pandas as pd

df = pd.read_csv('insurance_charges.csv')
df.shape

(500, 7)

In [2]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,22,male,32.6,5,no,southeast,8471.19
1,46,male,25.0,3,no,southwest,7697.12
2,46,male,19.7,5,no,southeast,6606.67
3,19,male,31.2,4,no,northeast,8769.66
4,61,female,21.0,1,no,southwest,6232.56


In [3]:
df.dtypes

age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object

In [4]:
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [5]:
df.describe()

,age,bmi,children,charges
count,500.000000,500.000000,500.000000,500.000000
mean,41.208000,29.851800,2.594000,14159.262820
std,13.411356,5.810482,1.680122,14244.392494
min,18.000000,16.000000,0.000000,3310.140000
25%,29.000000,25.600000,1.000000,6247.147500
50%,42.000000,30.200000,3.000000,7630.510000
75%,52.000000,33.900000,4.000000,10315.352500
max,64.000000,46.700000,5.000000,92810.370000


### Target: `charges`

Check the shape of the cost distribution.

In [6]:
print('median:', df['charges'].median())
print('mean  :', df['charges'].mean())
print('skew  :', round(df['charges'].skew(), 2))
df['charges'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])

median: 7630.51
mean  : 14159.26282
skew  : 1.81


0.10     5338.7440
0.25     6247.1475
0.50     7630.5100
0.75    10315.3525
0.90    40094.8290
0.99    52701.7041
Name: charges, dtype: float64

In [7]:
for c in ['sex', 'smoker', 'region']:
    print(c)
    print(df[c].value_counts(), '\n')

sex
sex
female    257
male      243
Name: count, dtype: int64 

smoker
smoker
no     398
yes    102
Name: count, dtype: int64 

region
region
southwest    135
northeast    130
northwest    122
southeast    113
Name: count, dtype: int64 



### What drives `charges`?

In [8]:
df.groupby('smoker')['charges'].agg(['count', 'median', 'mean'])

,count,median,mean
smoker,,,
no,398,7082.33,7304.190176
yes,102,39997.27,40907.487451


In [9]:
df.corr(numeric_only=True)['charges'].sort_values(ascending=False)

charges     1.000000
bmi         0.136461
age         0.078564
children    0.065376
Name: charges, dtype: float64

### Notes

- `charges` is continuous, strictly positive, and clearly **right-skewed** (mean above median, long upper tail).
- The distribution is **bimodal**: `smoker` splits it into a low-cost and a high-cost group, smokers cost several times more.
- `age` and `bmi` are positively correlated with `charges`; `children` weakly. `sex` and `region` look minor.
- No missing values. Mixed types: numeric (`age`, `bmi`, `children`) + categorical (`sex`, `smoker`, `region`), all low-cardinality. `n = 500`, `p = 6`.
- Likely modeling angle: regression on a positive, skewed target with a strong `smoker x bmi` interaction. Probably evaluate with RMSE/MAE on cost.